In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import csv
import math
import os
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import openpyxl
import datetime
from scipy.stats import lognorm
import re
import string
from bs4 import BeautifulSoup
import requests
import unicodedata # for removing accented characters
import icecream as ic

import pdfplumber


In [2]:
# 100m/200m/400m events


file = "/Users/veesheenyuen/Desktop/DataScience/SAA/3rd Asian Youth Games/Reports_BAYG2025_ATH/ATH.W.100M--------------.SFNL.000100--_C73A_Results.pdf"
df_table=pd.DataFrame()  # initialize empty master df

df=None

pattern = re.compile(
    r"^(?P<RANK>\d+)\s+"                                # RANK
    r"(?P<BIB>\d+)\s+"                                  # BIB
    r"(?P<NAME>.+?)\s+"                                 # NAME (greedy, up to NOC)
    r"(?P<NATION>[A-Z]{3})\s+"                          # NATION
    r"(?P<DOB>\d{2}\s[A-Z]{3}\s\d{4})\s+"               # DOB
    r"(?P<LANE>\d+)\s+"                                 # LANE
    r"(?P<RX_TIME>\d+\.\d+)\s+"                         # RX_TIME
    r"(?P<RESULT>\d+\.\d+)"                             # RESULT
    r"(?:\s+(?P<QUALIFICATION>[A-Za-z]))?"              # Optional QUALIFICATION (single letter, upper/lower)
    r"(?:\s+(?P<REMARKS>[A-Z]{1,}(?:\s[A-Z]{1,})*))?$"  # Optional REMARKS (one or more uppercase, possibly spaced)
)


with pdfplumber.open(file) as pdf:
        
    for i in range(len(pdf.pages)):    
        
        page = pdf.pages[i]  # can iterate over different pages
        table=page.extract_table()
        
        print('TABLE1', table)
        text=page.extract_text()        
        
        df=pd.DataFrame(text.splitlines())
        
                        
        for index, row in df.iterrows(): # find row containing event details and column names


            line = row[0]  # Access the string text from the row Series
            
            match = re.match(pattern, line)
            if match:
                parsed = match.groupdict()
                print(parsed)
            else:
                print("No match found")



TABLE1 None
No match found
No match found
No match found
No match found
No match found
No match found
No match found
No match found
No match found
No match found
No match found
No match found
No match found
No match found
{'RANK': '1', 'BIB': '517', 'NAME': 'SALEM Dana', 'NATION': 'QAT', 'DOB': '07 OCT 2009', 'LANE': '4', 'RX_TIME': '0.211', 'RESULT': '11.70', 'QUALIFICATION': 'Q', 'REMARKS': 'OYGBM'}
{'RANK': '2', 'BIB': '419', 'NAME': 'YAN Xinyi', 'NATION': 'CHN', 'DOB': '22 NOV 2009', 'LANE': '5', 'RX_TIME': '0.182', 'RESULT': '11.87', 'QUALIFICATION': 'Q', 'REMARKS': None}
{'RANK': '3', 'BIB': '567', 'NAME': 'SAFEENAH Laila', 'NATION': 'UAE', 'DOB': '28 SEP 2009', 'LANE': '3', 'RX_TIME': '0.148', 'RESULT': '11.89', 'QUALIFICATION': 'Q', 'REMARKS': None}
{'RANK': '4', 'BIB': '548', 'NAME': 'WATARUJIKRIT Nattawadee', 'NATION': 'THA', 'DOB': '08 MAR 2009', 'LANE': '6', 'RX_TIME': '0.157', 'RESULT': '12.12', 'QUALIFICATION': 'q', 'REMARKS': 'PB'}
{'RANK': '5', 'BIB': '554', 'NAME': 'CH

In [9]:
# Shotput

file = "/Users/veesheenyuen/Desktop/DataScience/SAA/3rd Asian Youth Games/Reports_BAYG2025_ATH/ATH.M.SHOTPUT-----------.FNL-.000100--_C73S_Results.pdf"

pattern = re.compile(
    r"^(?P<RANK>\d+)\s+"                                       # 1. RANK: digits at start
    r"(?P<BIB>\d+)\s+"                                         # 2. BIB: digits
    r"(?P<NAME>.+?)\s+"                                        # 3. NAME: anything up to NOC code
    r"(?P<NOC>[A-Z]{3})\s+"                                    # 4. NATIONALITY (NOC code)
    r"(?P<DOB>\d{2}\s[A-Z]{3}\s\d{4})\s+"                      # 5. DOB (dd MMM yyyy)
    r"(?P<ATT1>[A-Z\d\.]+)\s+"                                 # 6. Attempt 1 (number or X)
    r"(?P<ATT2>[A-Z\d\.]+)\s+"                                 # 7. Attempt 2
    r"(?P<ATT3>[A-Z\d\.]+)\s+"                                 # 8. Attempt 3
    r"(?P<ATT4>[A-Z\d\.]+)\s+"                                 # 9. Attempt 4
    r"(?P<ATT5>[A-Z\d\.]+)\s+"                                 # 10. Attempt 5
    r"(?P<ATT6>[A-Z\d\.]+)\s+"                                 # 11. Attempt 6
    r"(?P<RESULT>[A-Z\d\.]+)"                                  # 12. Result (final column)
    r"(?:\s+(?P<REMARKS>[A-Z]+))?"                            # 13. Optional Remarks (PB/SB/etc)
)

with pdfplumber.open(file) as pdf:
        
    for i in range(len(pdf.pages)):    
        
        page = pdf.pages[i]  # can iterate over different pages
        table=page.extract_table()
        
        print('TABLE1', table)
        text=page.extract_text()        
        
        df=pd.DataFrame(text.splitlines())
        
                        
        for index, row in df.iterrows(): # find row containing event details and column names


            line = row[0]  # Access the string text from the row Series
            
            match = pattern.match(line)
            if match:
                parsed = match.groupdict()
                # Zip attempts into dictionary
                attempts = {str(i+1): parsed[f'ATT{i+1}'] for i in range(6)}
                # Compose the final record for your DataFrame
                record = {
                "RANK": parsed["RANK"],
                "BIB": parsed["BIB"],
                "NAME": parsed["NAME"],
                "NATIONALITY": parsed["NOC"],
                "DOB": parsed["DOB"],
                "RESULT": parsed["RESULT"],
                "REMARKS": parsed.get("REMARKS"),
                "DICTIONARY": attempts
                }
                print(record)
            else:
                print("No match")


TABLE1 [['Starting Order', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11'], ['Initial', '113', '176', '219', '177', '280', '167', '139', '272', '189', '108', '234'], ['Final', '272', '280', '219', '177', '139', '167', '176', '113', '', '', '']]
No match
No match
No match
No match
No match
No match
No match
No match
No match
No match
No match
No match
No match
{'RANK': '1', 'BIB': '113', 'NAME': 'JIANG Haozheng', 'NATIONALITY': 'CHN', 'DOB': '05 DEC 2009', 'RESULT': '18.36', 'REMARKS': None, 'DICTIONARY': {'1': 'X', '2': '18.36', '3': 'X', '4': 'X', '5': 'X', '6': 'X'}}
{'RANK': '2', 'BIB': '176', 'NAME': 'CHOI Jiho', 'NATIONALITY': 'KOR', 'DOB': '13 SEP 2010', 'RESULT': '17.13', 'REMARKS': None, 'DICTIONARY': {'1': '16.31', '2': '16.33', '3': '17.05', '4': '17.13', '5': '17.01', '6': '17.06'}}
{'RANK': '3', 'BIB': '280', 'NAME': 'SRIWIYA Chaituch', 'NATIONALITY': 'THA', 'DOB': '07 APR 2009', 'RESULT': '16.37', 'REMARKS': None, 'DICTIONARY': {'1': 'X', '2': '15.50', '3': 'X', '

In [8]:
# Triple Jump

file = "/Users/veesheenyuen/Desktop/DataScience/SAA/3rd Asian Youth Games/Reports_BAYG2025_ATH/ATH.M.TRPLJUMP----------.FNL-.000100--_C73P_Results.pdf"
rows = []

athlete_pattern = re.compile(
    r"^(?P<RANK>\d+)?\s+"                                 # Rank, sometimes missing (use ?)
    r"(?P<BIB>\d+)?\s+"                                   # Bib, sometimes missing (use ?)
    r"(?P<NAME>[A-Z\'\- ]+[A-Za-z\'\- ]*)\s+"             # Name (all caps at start, handle spaces/hyphens, tolerate no NOC in some rows)
    r"(?P<NOC>[A-Z]{3})\s+"
    r"(?P<DOB>\d{2}\s[A-Z]{3}\s\d{4})\s+"
    r"(?P<ATT1>[A-Za-z\d\.]+)\s+"
    r"(?P<ATT2>[A-Za-z\d\.]+)\s+"
    r"(?P<ATT3>[A-Za-z\d\.]+)\s+"
    r"(?P<ATT4>[A-Za-z\d\.]+)\s+"
    r"(?P<ATT5>[A-Za-z\d\.]+)\s+"
    r"(?P<ATT6>[A-Za-z\d\.]+)\s+"
    r"(?P<RESULT>\d+\.\d+)"
    r"(?:\s+(?P<REMARKS>[A-Z]+))?"
    r"$"
)

wind_pattern = re.compile(
    r"^\s*"
    r"(?P<W1>[+\-\d\.]+)\s+"
    r"(?P<W2>[+\-\d\.]+)\s+"
    r"(?P<W3>[+\-\d\.]+)\s+"
    r"(?P<W4>[+\-\d\.]+)\s+"
    r"(?P<W5>[+\-\d\.]+)\s+"
    r"(?P<W6>[+\-\d\.]+)"
    r".*$"
)

rows = []

with pdfplumber.open(file) as pdf:
    for page in pdf.pages:
        text = page.extract_text()
        if not text: continue
        lines = text.splitlines()
        i = 0
        while i < len(lines) - 1:
            match_a = athlete_pattern.match(lines[i])
            match_w = wind_pattern.match(lines[i+1])
            if match_a and match_w:
                attempts = [match_a.group(f'ATT{j+1}') for j in range(6)]
                winds = [match_w.group(f'W{j+1}') for j in range(6)]
                zipped = {str(j+1): {'value': attempts[j], 'wind': winds[j]} for j in range(6)}
                record = {
                    "RANK": match_a.group("RANK"),
                    "BIB": match_a.group("BIB"),
                    "NAME": match_a.group("NAME").strip(),
                    "NATIONALITY": match_a.group("NOC"),
                    "DOB": match_a.group("DOB"),
                    "RESULT": match_a.group("RESULT"),
                    "REMARKS": match_a.group("REMARKS"),
                    "DICTIONARY": zipped
                }
                rows.append(record)
                i += 2  # Move to next athlete (skip wind line)
            else:
                i += 1

df = pd.DataFrame(rows)
print(df.head())


  RANK  BIB           NAME NATIONALITY          DOB RESULT REMARKS  \
0    3  163  KLEPININ Gleb         KAZ  30 NOV 2009  14.38    None   

                                          DICTIONARY  
0  {'1': {'value': 'X', 'wind': '-0.8'}, '2': {'v...  


In [7]:
df

,RANK,BIB,NAME,NATIONALITY,DOB,RESULT,REMARKS,DICTIONARY
0,3,163,KLEPININ Gleb,KAZ,30 NOV 2009,14.38,None,"{'1': {'value': 'X', 'wind': '-0.8'}, '2': {'v..."
